# Step 1: Household environmental impact determination by consumption category.

Before running this notebook, please refer to the [Step 1 README](../Step_1/README.md) for instructions on the required input matrices and folder structure.

This notebook is organized into three main sections:
1. Loading the data files
2. Performing calculations
3. Aggregating the results into consumption categories

## Case study

- **Region:** Province of Quebec, Canada
- **Year:** 2021
- **Impact studied:** Climate change short-term

OpenIO-Canada was used to create the Input-Output (IO) tables, utilizing the Canadian Supply and Use Tables (SUT) from 2021 and the product-by-product (pxp) Input-Output Tables (IOT) from EXIOBASE for the same year. The tables were generated without capital endogenization.

Sources:

- [OpenIO- Canada v2.9](https://github.com/CIRAIG/OpenIO-Canada)
- [Canadian SUT](https://www150.statcan.gc.ca/n1/en/catalogue/15-602-X)
- [EXIOBASE IOT pxp](https://zenodo.org/records/5589597)

## Required libraries

In [1]:
import pandas as pd
import numpy as np
import os

## 1. Loading the data files

### Leontief inverse - L

In [2]:
L = pd.read_csv('1_Loading_the_data_files/L_2021.csv', header=[0, 1], index_col=[0, 1])

### Final demand

The final demand is represented in two mathematical structures:

- **Y:** Contains the final demand of government, households and capital investment across all Canadian provinces.
- **Fy:** Contains the direct emissions associated with the final demand for all Canadian provinces.

These structures are specific to the OpenIO-Canada model and the Province of Quebec case study. For other regions or MRIO models, the filtering step may differ.

In [3]:
Y = pd.read_csv('1_Loading_the_data_files/Y_2021.csv', header=[0, 1, 2], index_col=[0, 1])

To analyze final demand specific to households in the Province of Quebec, it is necessary to filter the data: 

In [4]:
sector = ('CA-QC','Household final consumption expenditure')

In [5]:
Y_CA_QC= Y.loc[:, sector]

To be able to analyze the contribution of each IO sector, it is necessary to diagonalize the vector:

In [6]:
diag_Y_CA_QC = pd.DataFrame(np.diag(Y_CA_QC.sum(1)),  index=Y.index, columns=Y.index)

In [7]:
Fy = pd.read_csv('1_Loading_the_data_files/FY_2021.csv', header=[0, 1, 2], index_col=[0])

### Environmental extensions matrix - S

In [8]:
S = pd.read_csv('1_Loading_the_data_files/S_2021.csv', header=[0, 1], index_col=[0])

### Characterization matrix - C
This matrix is built using IMPACT World+ (IW+) v2.1 method

In [9]:
C = pd.read_csv('1_Loading_the_data_files/C_2021.csv', header=[0], index_col=[0, 1])

## 2. Performing calculations

In [10]:
contribution_analysis_households_CA_QC = C.dot(S).dot(L).dot(diag_Y_CA_QC)

Now it is necessary to add the direct emissions Fy: 

In [11]:
direct_emissions = C.dot(Fy)[C.dot(Fy)!=0].dropna(how='all',axis=1).fillna(0)
direct_emissions.columns = direct_emissions.columns.droplevel(1)
direct_emissions.columns = pd.MultiIndex.from_product([direct_emissions.columns.levels[0], [
    "Private vehicle use","Heating and cooking","Water use by households"]])

In [12]:
contribution_analysis_households_CA_QCt = pd.concat([contribution_analysis_households_CA_QC, 
                                              direct_emissions.drop([
                                                  i for i in direct_emissions.columns.levels[0] if i != 'CA-QC'],axis=1)],
                                             axis=1)

Filtering results for the Climate Change (short term) impact category:

In [13]:
CC_Households_Quebec = contribution_analysis_households_CA_QCt.loc[('Climate change, short term','kg CO2 eq (short)')].T.sort_values(ascending=False)

Excluding sectors with zero Climate Change impacts and exporting the results

In [14]:
CC_Households_Quebec = CC_Households_Quebec[CC_Households_Quebec != 0]
CC_Households_Quebec.to_csv('CC_Households_Quebec_2021.csv')

In [15]:
TotalCO2emissions = CC_Households_Quebec.sum()

In [16]:
Quebec_Population= 8603995

Source of the Quebec population data for 2021 [Statistics Canada](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1710000901&cubeTimeFrame.startMonth=07&cubeTimeFrame.startYear=2021&cubeTimeFrame.endMonth=10&cubeTimeFrame.endYear=2021&referencePeriods=20210701%2C20211001)

In [17]:
CC_Quebec_per_capita= TotalCO2emissions/Quebec_Population
CC_Quebec_per_capita

9142.314651367573

According to the analysis, the climate change emissions associated with household consumption in the province of Quebec in 2021 were 9.1 tons of CO2 equivalent per capita.

## 3. Aggregating the results into consumption categories

To aggregate the IO sectors into consumption categories, it is first necessary to identify how many and which sectors are associated with Climate Change impacts from household consumption.


In [18]:
sectors_list = pd.DataFrame({'sectors': CC_Households_Quebec.index.get_level_values(1).unique()}).sort_values(by='sectors').reset_index(drop=True)
sectors_list

,sectors
0,"Accounting, tax preparation, bookkeeping and p..."
1,Admissions to live performing arts performances
2,Admissions to live sporting events
3,Admissions to motion picture film exhibitions
4,Advertising space in printed newspapers
...,...
415,"Wood products, n.e.c."
416,Wood waste for treatment: incineration
417,Wood waste for treatment: landfill
418,"Wool, silk-worm cocoons"


For interpretability, the remaining sectors are aggregated into ten consumption categories based on sector descriptions from the North American Product Classification System (NAPCS) and the Classification of Products by Activity (CPA): 
1. Food expenditures
2. Transportation
3. Shelter
4. Household operations, furnishing and equipment
5. Clothing and accessories
6. Health and personal care
7. Recreation
8. Tobacco, alcohol, non-medical cannabis and game of chance
9. Miscellaneous expenditures
10. Education and reading materials.

To aggregate it is necessary to load the concordance table made between the 420 IO sectors and the 10 consumption categories, and apply it to the climate change emissions by households

In [19]:
concordance_df = pd.read_excel('Aggregation.xlsx')

In [20]:
sector_mapping = dict(zip(concordance_df['IO Sector'], concordance_df['Consumption category']))

In [21]:
CC_Households_Quebec_mapped = CC_Households_Quebec.copy()
CC_Households_Quebec_mapped.index = pd.MultiIndex.from_arrays([
    CC_Households_Quebec.index.get_level_values(0),
    CC_Households_Quebec.index.get_level_values(1).map(sector_mapping)
])

In [22]:
CC_by_category = CC_Households_Quebec_mapped.groupby(level=1).sum()
CC_by_category

Clothing and accessories                                       4.569940e+09
Education and reading materials                                3.170731e+08
Food expenditures                                              1.803330e+10
Health and personal care                                       2.102858e+09
Household operations, furnishings and, equipment               8.832877e+09
Miscellaneous expenditures                                     1.269728e+09
Recreation                                                     1.370765e+09
Shelter                                                        1.149927e+10
Tobacco, alcohol, non-medical cannabis, and games of chance    2.409753e+09
Transportation                                                 2.817835e+10
Name: (Climate change, short term, kg CO2 eq (short)), dtype: float64

In [23]:
Total_CC = CC_by_category.sum()

In [24]:
CC_by_category_pct = (CC_by_category / Total_CC) * 100

In [25]:
CC_by_category_results = pd.concat([CC_by_category, CC_by_category_pct], axis=1, keys=['CC emissions (kg CO2 eq)', 'Percentage (%)'])

In [26]:
CC_by_category_results = CC_by_category_results.sort_values(by=('CC emissions (kg CO2 eq)'), ascending=False)

In [27]:
CC_by_category_results.to_excel('CC_by_consumption_category_2021.xlsx')
CC_by_category_results

,CC emissions (kg CO2 eq),Percentage (%)
Transportation,2.817835e+10,35.857656
Food expenditures,1.803330e+10,22.947826
Shelter,1.149927e+10,14.633113
"Household operations, furnishings and, equipment",8.832877e+09,11.240055
Clothing and accessories,4.569940e+09,5.815362
"Tobacco, alcohol, non-medical cannabis, and games of chance",2.409753e+09,3.066471
Health and personal care,2.102858e+09,2.675939
Recreation,1.370765e+09,1.744333
Miscellaneous expenditures,1.269728e+09,1.615761
Education and reading materials,3.170731e+08,0.403483
